### Dataset and Task Metadata

In [38]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="mic",
    dataset_year="2020",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/dataset/579/myocardial+infarction+complications",
    download_description="""
We download the data from the UCI repository and uzip it to a predefined folder.

mkdir -p local-data-warehouse/mic/ && wget -P local-data-warehouse/mic/ https://archive.ics.uci.edu/static/public/579/myocardial+infarction+complications.zip && unzip local-data-warehouse/mic/myocardial+infarction+complications.zip -d local-data-warehouse/mic/ && rm local-data-warehouse/mic/myocardial+infarction+complications.zip
""",

# References
academic_reference_bibtex="""@article{golovenkin2020trajectories,
  title={Trajectories, bifurcations, and pseudo-time in large clinical datasets: applications to myocardial infarction and diabetes data},
  author={Golovenkin, Sergey E and Bac, Jonathan and Chervov, Alexander and Mirkes, Evgeny M and Orlova, Yuliya V and Barillot, Emmanuel and Gorban, Alexander N and Zinovyev, Andrei},
  journal={GigaScience},
  volume={9},
  number={11},
  pages={giaa128},
  year={2020},
  publisher={Oxford University Press}
}
""",
    academic_reference_bibtex_key="golovenkin2020trajectories",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
 - Remove additional targets
 - Map binary features to Yes/No
 - Fill missing values with NaN and in numeric features replace them with mean and convert to int
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="LET_IS",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="LET_IS",
)

## Preprocessing

In [40]:
import numpy as np
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/MI.data", header=None)

# Concatenate the two datasets
feature_names = [
    "ID",
    "AGE",              # int
    "SEX",              # {0, 1}
    "INF_ANAM",         # {0.0, 1.0, 2.0, 3.0, nan}
    "STENOK_AN",        # {0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, nan}
    "FK_STENOK",        # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "IBS_POST",         # {0.0, 1.0, 2.0, nan}
    "IBS_NASL",         # {0.0, 1.0, nan}
    "GB",               # {0.0, 1.0, 2.0, 3.0, nan}
    "SIM_GIPERT",       # {0.0, 1.0, nan}
    "DLIT_AG",          # {0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, nan}
    "ZSN_A",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "nr_11",            # {0.0, 1.0, nan}
    "nr_01",            # {0.0, 1.0, nan}
    "nr_02",            # {0.0, 1.0, nan}
    "nr_03",            # {0.0, 1.0, nan}
    "nr_04",            # {0.0, 1.0, nan}
    "nr_07",            # {0.0, 1.0, nan}
    "nr_08",            # {0.0, 1.0, nan}
    "np_01",            # {0.0, 1.0, nan}
    "np_04",            # {0.0, 1.0, nan}
    "np_05",            # {0.0, 1.0, nan}
    "np_07",            # {0.0, 1.0, nan}
    "np_08",            # {0.0, 1.0, nan}
    "np_09",            # {0.0, 1.0, nan}
    "np_10",            # {0.0, 1.0, nan}
    "endocr_01",            # {0.0, 1.0, nan}
    "endocr_02",            # {0.0, 1.0, nan}
    "endocr_03",            # {0.0, 1.0, nan}
    "zab_leg_01",            # {0.0, 1.0, nan}
    "zab_leg_02",            # {0.0, 1.0, nan}
    "zab_leg_03",            # {0.0, 1.0, nan}
    "zab_leg_04",            # {0.0, 1.0, nan}
    "zab_leg_06",            # {0.0, 1.0, nan}
    "S_AD_KBRIG",            # REAL
    "D_AD_KBRIG",            # REAL
    "S_AD_ORIT",            # REAL
    "D_AD_ORIT",            # REAL
    "O_L_POST",            # {0.0, 1.0, nan}
    "K_SH_POST",            # {0.0, 1.0, nan}
    "MP_TP_POST",            # {0.0, 1.0, nan}
    "SVT_POST",            # {0.0, 1.0, nan}
    "GT_POST",            # {0.0, 1.0, nan}
    "FIB_G_POST",            # {0.0, 1.0, nan}
    "ant_im",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "lat_im",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "inf_im",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "post_im",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "IM_PG_P",            # {0.0, 1.0, nan}
    "ritm_ecg_p_01",            # {0.0, 1.0, nan}
    "ritm_ecg_p_02",            # {0.0, 1.0, nan}
    "ritm_ecg_p_04",            # {0.0, 1.0, nan}
    "ritm_ecg_p_06",            # {0.0, 1.0, nan}
    "ritm_ecg_p_07",            # {0.0, 1.0, nan}
    "ritm_ecg_p_08",            # {0.0, 1.0, nan}
    "n_r_ecg_p_01",            # {0.0, 1.0, nan}
    "n_r_ecg_p_02",            # {0.0, 1.0, nan}
    "n_r_ecg_p_03",            # {0.0, 1.0, nan}
    "n_r_ecg_p_04",            # {0.0, 1.0, nan}
    "n_r_ecg_p_05",            # {0.0, 1.0, nan}
    "n_r_ecg_p_06",            # {0.0, 1.0, nan}
    "n_r_ecg_p_08",            # {0.0, 1.0, nan}
    "n_r_ecg_p_09",            # {0.0, 1.0, nan}
    "n_r_ecg_p_10",            # {0.0, 1.0, nan}
    "n_p_ecg_p_01",            # {0.0, 1.0, nan}
    "n_p_ecg_p_03",            # {0.0, 1.0, nan}
    "n_p_ecg_p_04",            # {0.0, 1.0, nan}
    "n_p_ecg_p_05",            # {0.0, 1.0, nan}
    "n_p_ecg_p_06",            # {0.0, 1.0, nan}
    "n_p_ecg_p_07",            # {0.0, 1.0, nan}
    "n_p_ecg_p_08",            # {0.0, 1.0, nan}
    "n_p_ecg_p_09",            # {0.0, 1.0, nan}
    "n_p_ecg_p_10",            # {0.0, 1.0, nan}
    "n_p_ecg_p_11",            # {0.0, 1.0, nan}
    "n_p_ecg_p_12",            # {0.0, 1.0, nan}
    "fibr_ter_01",            # {0.0, 1.0, nan}
    "fibr_ter_02",            # {0.0, 1.0, nan}
    "fibr_ter_03",            # {0.0, 1.0, nan}
    "fibr_ter_05",            # {0.0, 1.0, nan}
    "fibr_ter_06",            # {0.0, 1.0, nan}
    "fibr_ter_07",            # {0.0, 1.0, nan}
    "fibr_ter_08",            # {0.0, 1.0, nan}
    "GIPO_K",            # {0.0, 1.0, nan}
    "K_BLOOD",            # REAL
    "GIPER_NA",            # {0.0, 1.0, nan}
    "NA_BLOOD",            # REAL
    "ALT_BLOOD",            # REAL
    "AST_BLOOD",            # REAL
    "KFK_BLOOD",            # REAL
    "L_BLOOD",            # REAL
    "ROE",            # REAL
    "TIME_B_S",            # {1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, nan}
    "R_AB_1_n",            # {0.0, 1.0, 2.0, 3.0, nan}
    "R_AB_2_n",            # {0.0, 1.0, 2.0, 3.0, nan}
    "R_AB_3_n",            # {0.0, 1.0, 2.0, 3.0, nan}
    "NA_KB",            # {0.0, 1.0, nan}
    "NOT_NA_KB",            # {0.0, 1.0, nan}
    "LID_KB",            # {0.0, 1.0, nan}
    "NITR_S",            # {0.0, 1.0, nan}
    "NA_R_1_n",            # REAL
    "NA_R_2_n",            # REAL
    "NA_R_3_n",            # REAL
    "NOT_NA_1_n",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "NOT_NA_2_n",            # REAL
    "NOT_NA_3_n",            # REAL
    "LID_S_n",            # {0.0, 1.0, nan}
    "B_BLOK_S_n",            # {0.0, 1.0, nan}
    "ANT_CA_S_n",            # {0.0, 1.0, nan}
    "GEPAR_S_n",            # {0.0, 1.0, nan}
    "ASP_S_n",            # {0.0, 1.0, nan}
    "TIKL_S_n",            # {0.0, 1.0, nan}
    "TRENT_S_n",            # {0.0, 1.0, nan}
    "FIBR_PREDS",       # Atrial fibrillation		no
    "PREDS_TAH",        # Supraventricular tachycardia		no
    "JELUD_TAH",        # Ventricular tachycardia		no
    "FIBR_JELUD",        # Ventricular fibrillation		no
    "A_V_BLOK",        # Third-degree AV block		no
    "OTEK_LANC",        # Pulmonary edema		no
    "RAZRIV",        # Myocardial rupture		no
    "DRESSLER",        # Dressler syndrome		no
    "ZSN",        # Chronic heart failure		no
    "REC_IM",        # Relapse of the myocardial infarction		no
    "P_IM_STEN",        # Post-infar
    "LET_IS",            # {alive, asystole, cardiogenic_shock, myocardial_rupture, progress_congestive_heart_failure, pulmonary_edema, thromboembolism, ventricular_fibrillation}
]

df.columns = feature_names

df = df.drop(columns=["FIBR_PREDS", "PREDS_TAH", "JELUD_TAH", "FIBR_JELUD", "A_V_BLOK", "OTEK_LANC", "RAZRIV", "DRESSLER", "ZSN", "REC_IM", "P_IM_STEN"])

cat_features = [
    "ID",
    "SEX",              # {0, 1}
    "INF_ANAM",         # {0.0, 1.0, 2.0, 3.0, nan}
    "STENOK_AN",        # {0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, nan}
    "FK_STENOK",        # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "IBS_POST",         # {0.0, 1.0, 2.0, nan}
    "IBS_NASL",         # {0.0, 1.0, nan}
    "GB",               # {0.0, 1.0, 2.0, 3.0, nan}
    "SIM_GIPERT",       # {0.0, 1.0, nan}
    "DLIT_AG",          # {0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, nan}
    "ZSN_A",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "nr_11",            # {0.0, 1.0, nan}
    "nr_01",            # {0.0, 1.0, nan}
    "nr_02",            # {0.0, 1.0, nan}
    "nr_03",            # {0.0, 1.0, nan}
    "nr_04",            # {0.0, 1.0, nan}
    "nr_07",            # {0.0, 1.0, nan}
    "nr_08",            # {0.0, 1.0, nan}
    "np_01",            # {0.0, 1.0, nan}
    "np_04",            # {0.0, 1.0, nan}
    "np_05",            # {0.0, 1.0, nan}
    "np_07",            # {0.0, 1.0, nan}
    "np_08",            # {0.0, 1.0, nan}
    "np_09",            # {0.0, 1.0, nan}
    "np_10",            # {0.0, 1.0, nan}
    "endocr_01",            # {0.0, 1.0, nan}
    "endocr_02",            # {0.0, 1.0, nan}
    "endocr_03",            # {0.0, 1.0, nan}
    "zab_leg_01",            # {0.0, 1.0, nan}
    "zab_leg_02",            # {0.0, 1.0, nan}
    "zab_leg_03",            # {0.0, 1.0, nan}
    "zab_leg_04",            # {0.0, 1.0, nan}
    "zab_leg_06",            # {0.0, 1.0, nan}
    "O_L_POST",            # {0.0, 1.0, nan}
    "K_SH_POST",            # {0.0, 1.0, nan}
    "MP_TP_POST",            # {0.0, 1.0, nan}
    "SVT_POST",            # {0.0, 1.0, nan}
    "GT_POST",            # {0.0, 1.0, nan}
    "FIB_G_POST",            # {0.0, 1.0, nan}
    "ant_im",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "lat_im",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "inf_im",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "post_im",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "IM_PG_P",            # {0.0, 1.0, nan}
    "ritm_ecg_p_01",            # {0.0, 1.0, nan}
    "ritm_ecg_p_02",            # {0.0, 1.0, nan}
    "ritm_ecg_p_04",            # {0.0, 1.0, nan}
    "ritm_ecg_p_06",            # {0.0, 1.0, nan}
    "ritm_ecg_p_07",            # {0.0, 1.0, nan}
    "ritm_ecg_p_08",            # {0.0, 1.0, nan}
    "n_r_ecg_p_01",            # {0.0, 1.0, nan}
    "n_r_ecg_p_02",            # {0.0, 1.0, nan}
    "n_r_ecg_p_03",            # {0.0, 1.0, nan}
    "n_r_ecg_p_04",            # {0.0, 1.0, nan}
    "n_r_ecg_p_05",            # {0.0, 1.0, nan}
    "n_r_ecg_p_06",            # {0.0, 1.0, nan}
    "n_r_ecg_p_08",            # {0.0, 1.0, nan}
    "n_r_ecg_p_09",            # {0.0, 1.0, nan}
    "n_r_ecg_p_10",            # {0.0, 1.0, nan}
    "n_p_ecg_p_01",            # {0.0, 1.0, nan}
    "n_p_ecg_p_03",            # {0.0, 1.0, nan}
    "n_p_ecg_p_04",            # {0.0, 1.0, nan}
    "n_p_ecg_p_05",            # {0.0, 1.0, nan}
    "n_p_ecg_p_06",            # {0.0, 1.0, nan}
    "n_p_ecg_p_07",            # {0.0, 1.0, nan}
    "n_p_ecg_p_08",            # {0.0, 1.0, nan}
    "n_p_ecg_p_09",            # {0.0, 1.0, nan}
    "n_p_ecg_p_10",            # {0.0, 1.0, nan}
    "n_p_ecg_p_11",            # {0.0, 1.0, nan}
    "n_p_ecg_p_12",            # {0.0, 1.0, nan}
    "fibr_ter_01",            # {0.0, 1.0, nan}
    "fibr_ter_02",            # {0.0, 1.0, nan}
    "fibr_ter_03",            # {0.0, 1.0, nan}
    "fibr_ter_05",            # {0.0, 1.0, nan}
    "fibr_ter_06",            # {0.0, 1.0, nan}
    "fibr_ter_07",            # {0.0, 1.0, nan}
    "fibr_ter_08",            # {0.0, 1.0, nan}
    "GIPO_K",            # {0.0, 1.0, nan}
    "GIPER_NA",            # {0.0, 1.0, nan}
    "TIME_B_S",            # {1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, nan}
    "R_AB_1_n",            # {0.0, 1.0, 2.0, 3.0, nan}
    "R_AB_2_n",            # {0.0, 1.0, 2.0, 3.0, nan}
    "R_AB_3_n",            # {0.0, 1.0, 2.0, 3.0, nan}
    "NA_KB",            # {0.0, 1.0, nan}
    "NOT_NA_KB",            # {0.0, 1.0, nan}
    "LID_KB",            # {0.0, 1.0, nan}
    "NITR_S",            # {0.0, 1.0, nan}
    "NOT_NA_1_n",            # {0.0, 1.0, 2.0, 3.0, 4.0, nan}
    "LID_S_n",            # {0.0, 1.0, nan}
    "B_BLOK_S_n",            # {0.0, 1.0, nan}
    "ANT_CA_S_n",            # {0.0, 1.0, nan}
    "GEPAR_S_n",            # {0.0, 1.0, nan}
    "ASP_S_n",            # {0.0, 1.0, nan}
    "TIKL_S_n",            # {0.0, 1.0, nan}
    "TRENT_S_n",            # {0.0, 1.0, nan}
    "LET_IS",            # {alive, asystole, cardiogenic_shock, myocardial_rupture, progress_congestive_heart_failure, pulmonary_edema, thromboembolism, ventricular_fibrillation}
]

binary_features = [
    "IBS_NASL",         # {0.0, 1.0, nan}
    "SIM_GIPERT",       # {0.0, 1.0, nan}
    "nr_11",            # {0.0, 1.0, nan}
    "nr_01",            # {0.0, 1.0, nan}
    "nr_02",            # {0.0, 1.0, nan}
    "nr_03",            # {0.0, 1.0, nan}
    "nr_04",            # {0.0, 1.0, nan}
    "nr_07",            # {0.0, 1.0, nan}
    "nr_08",            # {0.0, 1.0, nan}
    "np_01",            # {0.0, 1.0, nan}
    "np_04",            # {0.0, 1.0, nan}
    "np_05",            # {0.0, 1.0, nan}
    "np_07",            # {0.0, 1.0, nan}
    "np_08",            # {0.0, 1.0, nan}
    "np_09",            # {0.0, 1.0, nan}
    "np_10",            # {0.0, 1.0, nan}
    "endocr_01",            # {0.0, 1.0, nan}
    "endocr_02",            # {0.0, 1.0, nan}
    "endocr_03",            # {0.0, 1.0, nan}
    "zab_leg_01",            # {0.0, 1.0, nan}
    "zab_leg_02",            # {0.0, 1.0, nan}
    "zab_leg_03",            # {0.0, 1.0, nan}
    "zab_leg_04",            # {0.0, 1.0, nan}
    "zab_leg_06",            # {0.0, 1.0, nan}
    "O_L_POST",            # {0.0, 1.0, nan}
    "K_SH_POST",            # {0.0, 1.0, nan}
    "MP_TP_POST",            # {0.0, 1.0, nan}
    "SVT_POST",            # {0.0, 1.0, nan}
    "GT_POST",            # {0.0, 1.0, nan}
    "FIB_G_POST",            # {0.0, 1.0, nan}
    "IM_PG_P",            # {0.0, 1.0, nan}
    "ritm_ecg_p_01",            # {0.0, 1.0, nan}
    "ritm_ecg_p_02",            # {0.0, 1.0, nan}
    "ritm_ecg_p_04",            # {0.0, 1.0, nan}
    "ritm_ecg_p_06",            # {0.0, 1.0, nan}
    "ritm_ecg_p_07",            # {0.0, 1.0, nan}
    "ritm_ecg_p_08",            # {0.0, 1.0, nan}
    "n_r_ecg_p_01",            # {0.0, 1.0, nan}
    "n_r_ecg_p_02",            # {0.0, 1.0, nan}
    "n_r_ecg_p_03",            # {0.0, 1.0, nan}
    "n_r_ecg_p_04",            # {0.0, 1.0, nan}
    "n_r_ecg_p_05",            # {0.0, 1.0, nan}
    "n_r_ecg_p_06",            # {0.0, 1.0, nan}
    "n_r_ecg_p_08",            # {0.0, 1.0, nan}
    "n_r_ecg_p_09",            # {0.0, 1.0, nan}
    "n_r_ecg_p_10",            # {0.0, 1.0, nan}
    "n_p_ecg_p_01",            # {0.0, 1.0, nan}
    "n_p_ecg_p_03",            # {0.0, 1.0, nan}
    "n_p_ecg_p_04",            # {0.0, 1.0, nan}
    "n_p_ecg_p_05",            # {0.0, 1.0, nan}
    "n_p_ecg_p_06",            # {0.0, 1.0, nan}
    "n_p_ecg_p_07",            # {0.0, 1.0, nan}
    "n_p_ecg_p_08",            # {0.0, 1.0, nan}
    "n_p_ecg_p_09",            # {0.0, 1.0, nan}
    "n_p_ecg_p_10",            # {0.0, 1.0, nan}
    "n_p_ecg_p_11",            # {0.0, 1.0, nan}
    "n_p_ecg_p_12",            # {0.0, 1.0, nan}
    "fibr_ter_01",            # {0.0, 1.0, nan}
    "fibr_ter_02",            # {0.0, 1.0, nan}
    "fibr_ter_03",            # {0.0, 1.0, nan}
    "fibr_ter_05",            # {0.0, 1.0, nan}
    "fibr_ter_06",            # {0.0, 1.0, nan}
    "fibr_ter_07",            # {0.0, 1.0, nan}
    "fibr_ter_08",            # {0.0, 1.0, nan}
    "GIPO_K",            # {0.0, 1.0, nan}
    "GIPER_NA",            # {0.0, 1.0, nan}
    "NA_KB",            # {0.0, 1.0, nan}
    "NOT_NA_KB",            # {0.0, 1.0, nan}
    "LID_KB",            # {0.0, 1.0, nan}
    "NITR_S",            # {0.0, 1.0, nan}
    "LID_S_n",            # {0.0, 1.0, nan}
    "B_BLOK_S_n",            # {0.0, 1.0, nan}
    "ANT_CA_S_n",            # {0.0, 1.0, nan}
    "GEPAR_S_n",            # {0.0, 1.0, nan}
    "ASP_S_n",            # {0.0, 1.0, nan}
    "TIKL_S_n",            # {0.0, 1.0, nan}
    "TRENT_S_n",            # {0.0, 1.0, nan}
]

num_features = feature_names = [
    "AGE",                  # REAL
    "S_AD_KBRIG",           # REAL
    "D_AD_KBRIG",           # REAL
    "S_AD_ORIT",            # REAL
    "D_AD_ORIT",            # REAL
    "K_BLOOD",              # REAL
    "NA_BLOOD",             # REAL
    "ALT_BLOOD",            # REAL
    "AST_BLOOD",            # REAL
    "KFK_BLOOD",            # REAL
    "L_BLOOD",              # REAL
    "ROE",                  # REAL
    "NA_R_1_n",             # REAL
    "NA_R_2_n",             # REAL
    "NA_R_3_n",             # REAL
    "NOT_NA_2_n",           # REAL
    "NOT_NA_3_n",           # REAL
]

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

df[cat_features] = df[cat_features].astype("category")
for col in binary_features:
    df[col] = df[col].map({0.0: "No", 1.0: "Yes"})
df = df.replace('?', np.nan)
df[num_features] = df[num_features].astype("float")
for col in num_features:
    df[col] = df[col].fillna(df[col].mean())
df[num_features] = df[num_features].astype("int")

/var/folders/x1/wbbfsghj623bqlfrnykvd7y80000gn/T/ipykernel_51680/1213405654.py:342: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df = df.replace('?', np.nan)


In [41]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,ID,AGE,SEX,INF_ANAM,STENOK_AN,FK_STENOK,IBS_POST,IBS_NASL,GB,SIM_GIPERT,DLIT_AG,ZSN_A,nr_11,nr_01,nr_02,nr_03,nr_04,nr_07,nr_08,np_01,np_04,np_05,np_07,np_08,np_09,np_10,endocr_01,endocr_02,endocr_03,zab_leg_01,zab_leg_02,zab_leg_03,zab_leg_04,zab_leg_06,S_AD_KBRIG,D_AD_KBRIG,S_AD_ORIT,D_AD_ORIT,O_L_POST,K_SH_POST,MP_TP_POST,SVT_POST,GT_POST,FIB_G_POST,ant_im,lat_im,inf_im,post_im,IM_PG_P,ritm_ecg_p_01,ritm_ecg_p_02,ritm_ecg_p_04,ritm_ecg_p_06,ritm_ecg_p_07,ritm_ecg_p_08,n_r_ecg_p_01,n_r_ecg_p_02,n_r_ecg_p_03,n_r_ecg_p_04,n_r_ecg_p_05,n_r_ecg_p_06,n_r_ecg_p_08,n_r_ecg_p_09,n_r_ecg_p_10,n_p_ecg_p_01,n_p_ecg_p_03,n_p_ecg_p_04,n_p_ecg_p_05,n_p_ecg_p_06,n_p_ecg_p_07,n_p_ecg_p_08,n_p_ecg_p_09,n_p_ecg_p_10,n_p_ecg_p_11,n_p_ecg_p_12,fibr_ter_01,fibr_ter_02,fibr_ter_03,fibr_ter_05,fibr_ter_06,fibr_ter_07,fibr_ter_08,GIPO_K,K_BLOOD,GIPER_NA,NA_BLOOD,ALT_BLOOD,AST_BLOOD,KFK_BLOOD,L_BLOOD,ROE,TIME_B_S,R_AB_1_n,R_AB_2_n,R_AB_3_n,NA_KB,NOT_NA_KB,LID_KB,NITR_S,NA_R_1_n,NA_R_2_n,NA_R_3_n,NOT_NA_1_n,NOT_NA_2_n,NOT_NA_3_n,LID_S_n,B_BLOK_S_n,ANT_CA_S_n,GEPAR_S_n,ASP_S_n,TIKL_S_n,TRENT_S_n,LET_IS
0,1493,67,0,1,0,0,2,NaN,2,NaN,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170,100,130,90,NaN,NaN,NaN,NaN,NaN,NaN,4,2,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,133,0,0,2,9,23,7,0,0,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
1,116,59,0,0,0,0,0,NaN,2,NaN,6,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136,81,134,82,NaN,NaN,NaN,NaN,NaN,NaN,4,2,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,136,0,0,2,8,15,8,0,0,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,136,58,1,0,0,0,2,NaN,0,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136,81,134,82,NaN,NaN,NaN,NaN,NaN,NaN,0,0,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,136,0,0,2,8,13,9,0,0,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,353,61,1,0,0,0,2,NaN,2,NaN,7,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136,81,100,70,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,136,0,0,2,7,7,5,0,0,0,NaN,NaN,NaN,NaN,0,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1303,77,0,0,0,0,0,NaN,0,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136,81,140,90,NaN,NaN,NaN,NaN,NaN,NaN,0,1,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,138,0,0,2,7,14,2,0,0,0,NaN,NaN,NaN,NaN,2,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


## Data Checks

In [42]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,700
Columns: 113
Use sampling: False (sample size: 1,700)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['ID', 'AGE', 'ROE', 'NA_BLOOD', 'S_AD_ORIT', 'S_AD_KBRIG', 'L_BLOOD', 'D_AD_KBRIG', 'D_AD_ORIT', 'TIME_B_S']
Rows remaining as candidates after top-10 filter: 0 (of 1,700)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 76 (67.26% of columns)
Duplicate column names:
  - SIM_GIPERT
  - nr_11
  - nr_01
  - nr_02
  - nr_03
  - nr_04
  - nr_07
  - nr_08
  - np_01
  - np_04
  - np_05
  - np_07
  - np_08
  - np_09
  - np_10
  - endocr_01
  - endocr_02
  - endocr_03
  - zab_leg_01
  - zab_leg_02
  - zab_leg_03
  - zab_leg_04
  - zab_leg_06
  - O_L_POST
  - K_SH_POST
  - MP_TP_POST
  - SVT_POST
  - GT_POST
  - FIB_G_POST
  - IM_PG_P
  - ritm_ecg_p_01
  - ritm_ecg_p_02
  - ritm_ecg_p_04
  - ritm_ecg_p_06
  - rit

In [43]:
# Sample Rows
df_head

,ID,AGE,SEX,INF_ANAM,STENOK_AN,FK_STENOK,IBS_POST,IBS_NASL,GB,SIM_GIPERT,DLIT_AG,ZSN_A,nr_11,nr_01,nr_02,nr_03,nr_04,nr_07,nr_08,np_01,np_04,np_05,np_07,np_08,np_09,np_10,endocr_01,endocr_02,endocr_03,zab_leg_01,zab_leg_02,zab_leg_03,zab_leg_04,zab_leg_06,S_AD_KBRIG,D_AD_KBRIG,S_AD_ORIT,D_AD_ORIT,O_L_POST,K_SH_POST,MP_TP_POST,SVT_POST,GT_POST,FIB_G_POST,ant_im,lat_im,inf_im,post_im,IM_PG_P,ritm_ecg_p_01,ritm_ecg_p_02,ritm_ecg_p_04,ritm_ecg_p_06,ritm_ecg_p_07,ritm_ecg_p_08,n_r_ecg_p_01,n_r_ecg_p_02,n_r_ecg_p_03,n_r_ecg_p_04,n_r_ecg_p_05,n_r_ecg_p_06,n_r_ecg_p_08,n_r_ecg_p_09,n_r_ecg_p_10,n_p_ecg_p_01,n_p_ecg_p_03,n_p_ecg_p_04,n_p_ecg_p_05,n_p_ecg_p_06,n_p_ecg_p_07,n_p_ecg_p_08,n_p_ecg_p_09,n_p_ecg_p_10,n_p_ecg_p_11,n_p_ecg_p_12,fibr_ter_01,fibr_ter_02,fibr_ter_03,fibr_ter_05,fibr_ter_06,fibr_ter_07,fibr_ter_08,GIPO_K,K_BLOOD,GIPER_NA,NA_BLOOD,ALT_BLOOD,AST_BLOOD,KFK_BLOOD,L_BLOOD,ROE,TIME_B_S,R_AB_1_n,R_AB_2_n,R_AB_3_n,NA_KB,NOT_NA_KB,LID_KB,NITR_S,NA_R_1_n,NA_R_2_n,NA_R_3_n,NOT_NA_1_n,NOT_NA_2_n,NOT_NA_3_n,LID_S_n,B_BLOK_S_n,ANT_CA_S_n,GEPAR_S_n,ASP_S_n,TIKL_S_n,TRENT_S_n,LET_IS
0,1493,67,0,1,0,0,2,NaN,2,NaN,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,170,100,130,90,NaN,NaN,NaN,NaN,NaN,NaN,4,2,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,133,0,0,2,9,23,7,0,0,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
1,116,59,0,0,0,0,0,NaN,2,NaN,6,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136,81,134,82,NaN,NaN,NaN,NaN,NaN,NaN,4,2,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,136,0,0,2,8,15,8,0,0,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,136,58,1,0,0,0,2,NaN,0,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136,81,134,82,NaN,NaN,NaN,NaN,NaN,NaN,0,0,2,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,136,0,0,2,8,13,9,0,0,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,353,61,1,0,0,0,2,NaN,2,NaN,7,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136,81,100,70,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,NaN,136,0,0,2,7,7,5,0,0,0,NaN,NaN,NaN,NaN,0,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,1303,77,0,0,0,0,0,NaN,0,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,136,81,140,90,NaN,NaN,NaN,NaN,NaN,NaN,0,1,4,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN,138,0,0,2,7,14,2,0,0,0,NaN,NaN,NaN,NaN,2,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [44]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,DLIT_AG,category,248.0,14.59,8.0,"0, 7, 6, 1, 5, 2, 3, 4"
1,R_AB_3_n,category,128.0,7.53,4.0,"0, 1, 2, 3"
2,TIME_B_S,category,126.0,7.41,9.0,"2, 9, 1, 3, 6, 7, 8, 5, 4"
3,R_AB_2_n,category,108.0,6.35,4.0,"0, 1, 2, 3"
4,STENOK_AN,category,106.0,6.24,7.0,"0, 6, 1, 2, 5, 3, 4"
5,ant_im,category,83.0,4.88,5.0,"0, 4, 1, 2, 3"
6,lat_im,category,80.0,4.71,5.0,"1, 0, 2, 3, 4"
7,inf_im,category,80.0,4.71,5.0,"0, 1, 2, 4, 3"
8,FK_STENOK,category,73.0,4.29,5.0,"2, 0, 3, 1, 4"
9,post_im,category,72.0,4.24,5.0,"0, 1, 2, 3, 4"


In [45]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
AGE,1700.0,61.852941,11.233548,26.0,92.0
IBS_NASL,0.0,NaN,NaN,NaN,NaN
SIM_GIPERT,0.0,NaN,NaN,NaN,NaN
nr_11,0.0,NaN,NaN,NaN,NaN
nr_01,0.0,NaN,NaN,NaN,NaN
nr_02,0.0,NaN,NaN,NaN,NaN
nr_03,0.0,NaN,NaN,NaN,NaN
nr_04,0.0,NaN,NaN,NaN,NaN
nr_07,0.0,NaN,NaN,NaN,NaN
nr_08,0.0,NaN,NaN,NaN,NaN


In [46]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column     rank                    
DLIT_AG    1        0    551  32.41
           2        7    432  25.41
           3     <NA>    248  14.59
           4        6    165   9.71
           5        1     93   5.47
FK_STENOK  1        2    854  50.24
           2        0    661  38.88
           3     <NA>     73   4.29
           4        3     54   3.18
           5        1     47   2.76
GB         1        2    880  51.76
           2        0    605  35.59
           3        3    195  11.47
           4        1     11   0.65
           5     <NA>      9   0.53
IBS_POST   1        2    683  40.18
           2        1    548  32.24
           3        0    418  24.59
           4     <NA>     51   3.00
ID         1        1      1   0.06
           2     1143      1   0.06
           3     1141      1   0.06
           4     1140      1   0.06
           5     1139      1   0.06
INF_ANAM   1        0   1060  62.35
           2        1    410  24.12
           3        2    147   8.65
           4        3     79   4.65
           5     <NA>      4   0.24
LET_IS     1        0   1429  84.06
           2        1    110   6.47
           3        3     54   3.18
           4        6     27   1.59
           5        7     27   1.59
NOT_NA_1_n 1        0   1237  72.76
           2        1    376  22.12
           3        2     53   3.12
           4        3     17   1.00
           5     <NA>     10   0.59
R_AB_1_n   1        0   1282  75.41
           2        1    298  17.53
           3        2     78   4.59
           4        3     26   1.53
           5     <NA>     16   0.94
R_AB_2_n   1        0   1414  83.18
           2        1    133   7.82
           3     <NA>    108   6.35
           4        2     44   2.59
           5        3      1   0.06
R_AB_3_n   1        0   1469  86.41
           2     <NA>    128   7.53
           3        1     86   5.06
           4        2     15   0.88
           5        3      2   0.12
SEX        1        1   1065  62.65
           2        0    635  37.35
STENOK_AN  1        0    661  38.88
           2        6    332  19.53
           3        1    146   8.59
           4        2    137   8.06
           5        5    125   7.35
TIME_B_S   1        2    360  21.18
           2        9    269  15.82
           3        1    198  11.65
           4        3    175  10.29
           5        6    151   8.88
ZSN_A      1        0   1468  86.35
           2        1    103   6.06
           3     <NA>     54   3.18
           4        3     29   1.71
           5        2     27   1.59
ant_im     1        0    660  38.82
           2        4    492  28.94
           3        1    392  23.06
           4     <NA>     83   4.88
           5        2     39   2.29
inf_im     1        0    937  55.12
           2        1    195  11.47
           3        2    191  11.24
           4        4    176  10.35
           5        3    121   7.12
lat_im     1        1    838  49.29
           2        0    576  33.88
           3        2     97   5.71
           4     <NA>     80   4.71
           5        3     72   4.24
post_im    1        0   1370  80.59
           2        1    157   9.24
           3     <NA>     72   4.24
           4        2     52   3.06
           5        3     35   2.06

In [47]:
# Target Distribution
target_df

,count,pct
LET_IS,,
0,1429,84.06
1,110,6.47
3,54,3.18
6,27,1.59
7,27,1.59
4,23,1.35
2,18,1.06
5,12,0.71


## Task Curation

In [48]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [49]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [50]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to mic/019d35b8-8932-76f6-9f6e-87c2587f0e6e
019d35b8-8932-76f6-9f6e-87c2587f0e6e
84ccc7b60391264594b2851a6b187c50b8b6500acf00a4409c3a3dc83a45fa03
